# Обучение Transformer-агента на Мульти-Ассет данных (BTC + ETH)

В этом ноутбуке мы тестируем сразу два крупных нововведения:
1. **`load_multi_crypto_data`**: Загружает данные сразу по нескольким монетам без дублирования кода.
2. **Multi-Asset `TradingEnvV5`**: Среда теперь принимает список датафреймов и при каждом `reset()` случайно выбирает монету.
3. **`GatedTransformerExtractor`**: Новая архитектура сети, которая обрабатывает `n_stack` свечей через механизм Self-Attention.

In [ ]:
import pandas as pd
import numpy as np
import os

from core.data.data_loader import load_multi_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from agents.ppo_agent import create_ppo_agent
from utils.experiment_manager import get_experiment_paths

In [ ]:
# 1. Настройка эксперимента
exp_name = "ppo_transformer_multi_asset"
model_path, norm_path, tb_log_dir = get_experiment_paths(exp_name)

print(f"Model will be saved to: {model_path}")
print(f"Normalizer will be saved to: {norm_path}")
print(f"Tensorboard logs: {tb_log_dir}")

In [ ]:
# 2. Загрузка мульти-ассет данных
dfs_dict = load_multi_crypto_data(symbols=["BTCUSDT", "ETHUSDT"], start_date="2023-01-01", interval="4h")

In [ ]:
# 3. Генерация фичей
fg = FeatureGenerator()

processed_dfs = []
for sym, df in dfs_dict.items():
    print(f"Генерация фичей для {sym}...")
    proc_df = fg.transform(df)
    processed_dfs.append(proc_df)
    print(f"{sym}: {len(proc_df)} свечей готовы.")

In [ ]:
# 4. Инициализация мульти-ассет среды
train_env = TradingEnvV5(df=processed_dfs, continuous_action=True, domain_randomization=True)

vec_env = DummyVecEnv([lambda: train_env])
vec_env = VecFrameStack(vec_env, n_stack=10)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

In [ ]:
# 5. Инициализация Трансформера
model = create_ppo_agent(
    vec_env,
    extractor_type="transformer",
    tensorboard_log=tb_log_dir
)

In [ ]:
# 6. Обучение
print(f"Запуск обучения {exp_name}...")
model.learn(total_timesteps=10000, progress_bar=True)

# 7. Сохранение результатов в папку эксперимента
model.save(model_path)
vec_env.save(norm_path)
print(f"✅ Эксперимент {exp_name} успешно сохранен!")